# Question Answering with LangChain, OpenAI, and MultiQuery Retriever

This interactive workbook demonstrates example of Elasticsearch's [MultiQuery Retriever](https://api.python.langchain.com/en/latest/retrievers/langchain.retrievers.multi_query.MultiQueryRetriever.html) to generate similar queries for a given user input and apply all queries to retrieve a larger set of relevant documents from a vectorstore.

Before we begin, we first split the fictional workplace documents into passages with `langchain` and uses OpenAI to transform these passages into embeddings and then store these into Elasticsearch.

We will then ask a question, generate similar questions using langchain and OpenAI, retrieve relevant passages from the vector store, and use langchain and OpenAI again to provide a summary for the questions.

## Install packages and import modules

In [1]:
!pip install -qU "langchain>=1.0" "langchain-core>=0.3" "langchain-community>=0.4" "langchain-classic>=0.3" langchain-openai langchain-elasticsearch tiktoken jq lark elasticsearch


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import os
from getpass import getpass
from langchain_openai.embeddings import OpenAIEmbeddings
#from langchain_elasticsearch import ElasticsearchStore
from langchain_openai import ChatOpenAI
from langchain_classic.retrievers.multi_query import MultiQueryRetriever  # ← CORRECT import for 1.0+


from langchain_community.vectorstores.elasticsearch import ElasticsearchStore
#from langchain_openai import OpenAIEmbeddings

# os.environ["OPENAI_API_KEY"] = getpass("OpenAI API Key: ")

## Connect to Elasticsearch

ℹ️ We're using an Elastic Cloud deployment of Elasticsearch for this notebook. If you don't have an Elastic Cloud deployment, sign up [here](https://cloud.elastic.co/registration?utm_source=github&utm_content=elasticsearch-labs-notebook) for a free trial.

We'll use the **Cloud ID** to identify our deployment, because we are using Elastic Cloud deployment. To find the Cloud ID for your deployment, go to https://cloud.elastic.co/deployments and select your deployment.

We will use [ElasticsearchStore](https://api.python.langchain.com/en/latest/vectorstores/langchain.vectorstores.elasticsearch.ElasticsearchStore.html) to connect to our elastic cloud deployment, This would help create and index data easily.  We would also send list of documents that we created in the previous step

In [ ]:
# https://www.elastic.co/search-labs/tutorials/install-elasticsearch/elastic-cloud#finding-your-cloud-id
ELASTIC_CLOUD_ID = getpass("Elastic Cloud ID: ")

# https://www.elastic.co/search-labs/tutorials/install-elasticsearch/elastic-cloud#creating-an-api-key
ELASTIC_API_KEY = getpass("Elastic Api Key: ")

# https://platform.openai.com/api-keys
OPENAI_API_KEY = getpass("OpenAI API key: ")


# Create OpenAI embedding model
embeddings = OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY)



/var/folders/zz/scd6zwnn6l79trs9xhwny74h0000gn/T/ipykernel_32140/2755091263.py:18: LangChainPendingDeprecationWarning: The class `ElasticsearchStore` will be deprecated in a future version. Use `Use class in langchain-elasticsearch package` instead.
  vectorstore = ElasticsearchStore(


In [4]:
# Set a meaningful index name
INDEX_NAME = "workplace-index" #give it a meaningful name,

# Create Elasticsearch vector store
vectorstore = ElasticsearchStore(
    es_cloud_id=ELASTIC_CLOUD_ID,
    es_api_key=ELASTIC_API_KEY,
    index_name=INDEX_NAME,
    embedding=embeddings  # ✅ Add this line
)


## Indexing Data into Elasticsearch
Let's download the sample dataset and deserialize the document.

In [5]:
from urllib.request import urlopen
import json

url = "https://raw.githubusercontent.com/elastic/elasticsearch-labs/main/example-apps/chatbot-rag-app/data/data.json"

response = urlopen(url)
data = json.load(response)

with open("temp.json", "w") as json_file:
    json.dump(data, json_file)

### Split Documents into Passages

We’ll chunk documents into passages in order to improve the retrieval specificity and to ensure that we can provide multiple passages within the context window of the final question answering prompt.

Here we are chunking documents into 800 token passages with an overlap of 400 tokens.

Here we are using a simple splitter but Langchain offers more advanced splitters to reduce the chance of context being lost.

In [6]:
from langchain_community.document_loaders import JSONLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import json  # Optional: for metadata extraction
from datetime import datetime # Import datetime here


def metadata_func(record: dict, metadata: dict) -> dict:
    #Populate the metadata dictionary with keys name, summary, url, category, and updated_at.
    metadata["name"] = record.get("name")
    metadata["summary"] = record.get("summary")
    metadata["url"] = record.get("url")
    metadata["category"] = record.get("category")
    metadata["updated_at"] = record.get("updated_at")
    return metadata



loader = JSONLoader(
    file_path="temp.json",
    jq_schema=".[]",  # Extracts array of records
    content_key="content",
    metadata_func=metadata_func,
)

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=1000,       # e.g., ~750 words
    chunk_overlap=200,     # Overlap for context preservation
)

docs = loader.load_and_split(text_splitter=text_splitter)


### Bulk Import Passages

Now that we have split each document into the chunk size of 800, we will now index data to elasticsearch using [ElasticsearchStore.from_documents](https://api.python.langchain.com/en/latest/vectorstores/langchain.vectorstores.elasticsearch.ElasticsearchStore.html#langchain.vectorstores.elasticsearch.ElasticsearchStore.from_documents).

We will use Cloud ID, Password and Index name values set in the `Create cloud deployment` step.

In [7]:
from datetime import datetime
from langchain_elasticsearch import ElasticsearchStore
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_classic.retrievers.multi_query import MultiQueryRetriever

# Clean docs metadata
clean_docs = []
for doc in docs:
    metadata = doc.metadata.copy()

    if metadata.get("updated_at") in ["", None, "null"]:
        metadata["updated_at"] = datetime.now().isoformat()

    # Use model_copy() instead of copy() to avoid Pydantic deprecation warning
    clean_docs.append(doc.model_copy(update={"metadata": metadata}))

# Create embeddings ONCE
embeddings = OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY)

# Create vectorstore ONCE using cleaned docs
vectorstore = ElasticsearchStore.from_documents(
    clean_docs,
    embeddings,
    index_name=INDEX_NAME,
    es_cloud_id=ELASTIC_CLOUD_ID,
    es_api_key=ELASTIC_API_KEY,
)

# Create LLM
llm = ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=0,
    openai_api_key=OPENAI_API_KEY # Explicitly pass the API key
)

# Create MultiQueryRetriever
retriever = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(search_kwargs={"k": 4}),
    llm=llm
)


# Question Answering with MultiQuery Retriever

Now that we have the passages stored in Elasticsearch, we can now ask a question to get the relevant passages.

In [8]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import chain as lc_chain
import logging

# Enable detailed logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Multi-query generator with 3 variants
MULTI_QUERY_PROMPT = ChatPromptTemplate.from_template("""
Generate 3 diverse versions of this question for better retrieval. Vary phrasing, keywords, and perspectives:

{question}

Queries (one per line):
""")

LLM_DOCUMENT_PROMPT = PromptTemplate.from_template("""
📄 [{source}]
{page_content}
---
""")

# Define the context prompt for the LLM
LLM_CONTEXT_PROMPT = ChatPromptTemplate.from_template("""
Answer the question based only on the following context:
{context}

Question: {question}
""")

def safe_combine_docs(docs):
    """Production-ready doc formatting with fallbacks"""
    doc_strings = []
    for i, doc in enumerate(docs):
        try:
            doc_dict = doc.model_dump()
            source = doc.metadata.get("name") or doc.metadata.get("source", f"Doc-{i}")
            doc_dict["source"] = source
            formatted = LLM_DOCUMENT_PROMPT.format(**doc_dict)
        except Exception as e:
            logger.warning(f"Doc format error: {e}")
            formatted = f"[Doc-{i}] {doc.page_content[:500]}..."
        doc_strings.append(formatted)
    return "\n\n".join(doc_strings)

# Self-healing chain: retry bad retrievals
def self_healing_retriever(query, max_tries=2):
    """Retry with rewritten query if empty results"""
    for attempt in range(max_tries):
        docs = retriever.invoke(query)
        if docs:
            return docs
        logger.info(f"Empty results (attempt {attempt+1}), rewriting...")
        query = llm.invoke(f"Rewrite for better retrieval: {query}").content
    return retriever.invoke(query)  # Fallback

_context = RunnableParallel(
    context=(RunnablePassthrough() | self_healing_retriever | safe_combine_docs),
    question=RunnablePassthrough(),
)

rag_chain = _context | LLM_CONTEXT_PROMPT | llm | StrOutputParser()

# Test with auto-multi-query
def multi_query_rag(question):
    """Generate + retrieve + answer"""

    query_chain = MULTI_QUERY_PROMPT | llm | StrOutputParser()

    generated_queries = query_chain.invoke({"question": question})

    print("\nGenerated Queries:")
    print("------------------")
    print(generated_queries)
    print("------------------\n")

    queries = [q.strip() for q in generated_queries.split("\n") if q.strip()]

    all_docs = []

    for q in queries:
        docs = self_healing_retriever(q)
        all_docs.extend(docs[:3])

    return rag_chain.invoke({
        "question": question,
        "context": safe_combine_docs(all_docs)
    })

print("---- Answer ----")

print(multi_query_rag("what is the nasa sales team?"))

---- Answer ----


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"



Generated Queries:
------------------
1. Can you provide information on the sales team at NASA?
2. What is the role of the sales team within NASA?
3. How does NASA's sales team operate and contribute to the organization's goals?
------------------



INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:langchain_classic.retrievers.multi_query:Generated queries: ['1. What details can you share about the sales department within NASA?', '2. Could you give me insights into the sales team operating within NASA?', "3. I'm interested in learning more about the sales team specifically within NASA. Can you provide relevant information?"]
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:elastic_transport.transport:POST https://8f16272b86e5488dae3f9d2a403f1070.us-central1.gcp.cloud.es.io:443/workplace-index/_search?_source_includes=metadata,text [status:200 duration:0.243s]
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:elastic_transport.transport:POST https://8f16272b86e5488dae3f9d2a403f1070.us-central1.gcp.cloud.es.io:443/workplace-index/_search?_source_includes=metadata,text [status:200 duration:0.227s]
INFO:httpx:

The NASA sales team consists of two Area Vice-Presidents: Laura Martinez for North America and Gary Johnson for South America.


**Generate at least two new iteratioins of the previous cells - Be creative.** Did you master Multi-
Query Retriever concepts through this lab?

Iteration 1: Comparative Multi-Query RAG with Source Tracking
This iteration asks multiple questions and compares answers side by side, while tracking which sources contributed to each answer.

In [10]:
from collections import Counter

def multi_query_rag_with_sources(question):
    """Enhanced RAG that tracks and displays source documents used"""
    
    query_chain = MULTI_QUERY_PROMPT | llm | StrOutputParser()
    generated_queries = query_chain.invoke({"question": question})
    
    queries = [q.strip() for q in generated_queries.split("\n") if q.strip()]
    
    all_docs = []
    source_tracker = Counter()
    
    print("\n🔍 Generated Queries:")
    print("─" * 40)
    for i, q in enumerate(queries, 1):
        print(f"{i}. {q}")
    print("─" * 40)
    
    for q in queries:
        docs = self_healing_retriever(q)
        for doc in docs[:3]:
            source = doc.metadata.get("name") or doc.metadata.get("source", "Unknown")
            source_tracker[source] += 1
            all_docs.append(doc)
    
    # Deduplicate docs by page_content
    seen = set()
    unique_docs = []
    for doc in all_docs:
        if doc.page_content not in seen:
            seen.add(doc.page_content)
            unique_docs.append(doc)
    
    print(f"\n📚 Sources Retrieved (hit count):")
    print("─" * 40)
    for source, count in source_tracker.most_common():
        print(f"  • {source}: {count} hit(s)")
    print("─" * 40)
    
    answer = rag_chain.invoke({
        "question": question,
        "context": safe_combine_docs(unique_docs)
    })
    
    return answer


# Run on multiple questions and compare
questions = [
    "what is the nasa sales team?",
    "what are the remote work policies?",
    "what employee benefits are available?"
]

for q in questions:
    print(f"\n{'='*50}")
    print(f"❓ Question: {q}")
    print(f"{'='*50}")
    answer = multi_query_rag_with_sources(q)
    print("\n---- Answer ----")
    print(answer)


❓ Question: what is the nasa sales team?


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"



🔍 Generated Queries:
────────────────────────────────────────
1. 1. Can you provide information on the sales team at NASA?
2. 2. What is the role of the sales team within NASA?
3. 3. How does NASA's sales team operate and contribute to the organization's goals?
────────────────────────────────────────


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:langchain_classic.retrievers.multi_query:Generated queries: ['1. What details can you share about the sales department within NASA?', '2. Could you give me insights into the sales team operating within NASA?', "3. I'm interested in learning more about the sales team specifically within NASA. Can you provide relevant information?"]
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:elastic_transport.transport:POST https://8f16272b86e5488dae3f9d2a403f1070.us-central1.gcp.cloud.es.io:443/workplace-index/_search?_source_includes=metadata,text [status:200 duration:0.509s]
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:elastic_transport.transport:POST https://8f16272b86e5488dae3f9d2a403f1070.us-central1.gcp.cloud.es.io:443/workplace-index/_search?_source_includes=metadata,text [status:200 duration:0.146s]
INFO:httpx:


📚 Sources Retrieved (hit count):
────────────────────────────────────────
  • Sales Organization Overview: 3 hit(s)
  • Fy2024 Company Sales Strategy: 3 hit(s)
  • Sales Engineering Collaboration: 3 hit(s)
────────────────────────────────────────


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:langchain_classic.retrievers.multi_query:Generated queries: ["What are the responsibilities of the NASA sales team's Area Vice-Presidents?", "How does the sales strategy document outline the objectives and action plans for the tech company's sales operations?", 'How can engineers effectively collaborate with the sales team in a tech company to ensure product success and customer satisfaction?']
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:elastic_transport.transport:POST https://8f16272b86e5488dae3f9d2a403f1070.us-central1.gcp.cloud.es.io:443/workplace-index/_search?_source_includes=metadata,text [status:200 duration:0.144s]
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:elastic_transport.transport:POST https://8f16272b86e5488dae3f9d2a403f1070.us-central1.gcp.cloud.es.io:443/workplace-index/_search?_sourc

... (9 outputs left out)

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:langchain_classic.retrievers.multi_query:Generated queries: ['1. Can you provide information on the employee benefits currently offered?', '2. What benefits are accessible to employees within the organization?', '3. Could you outline the various employee benefits that are available for staff members?']
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:elastic_transport.transport:POST https://8f16272b86e5488dae3f9d2a403f1070.us-central1.gcp.cloud.es.io:443/workplace-index/_search?_source_includes=metadata,text [status:200 duration:0.144s]
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:elastic_transport.transport:POST https://8f16272b86e5488dae3f9d2a403f1070.us-central1.gcp.cloud.es.io:443/workplace-index/_search?_source_includes=metadata,text [status:200 duration:0.143s]
INFO:httpx:HTTP Request: POST https://ap


📚 Sources Retrieved (hit count):
────────────────────────────────────────
  • New Employee Onboarding Guide: 3 hit(s)
  • April Work From Home Update: 3 hit(s)
  • Wfh Policy Update May 2023: 2 hit(s)
  • Company Vacation Policy: 1 hit(s)
────────────────────────────────────────


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:langchain_classic.retrievers.multi_query:Generated queries: ['1. What types of employee benefits can new employees access?', "2. How can new employees enroll in the company's benefits program?", '3. What steps should new employees take to update their tax information and documents?']
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:elastic_transport.transport:POST https://8f16272b86e5488dae3f9d2a403f1070.us-central1.gcp.cloud.es.io:443/workplace-index/_search?_source_includes=metadata,text [status:200 duration:0.142s]
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:elastic_transport.transport:POST https://8f16272b86e5488dae3f9d2a403f1070.us-central1.gcp.cloud.es.io:443/workplace-index/_search?_source_includes=metadata,text [status:200 duration:0.143s]
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/emb


---- Answer ----
Employees are eligible for various benefits, including health insurance, retirement plans, and paid time off.


Iteration 2: Confidence-Scored RAG with Fallback to Web Search Suggestion
This iteration adds a self-evaluation step where the LLM scores its own confidence and suggests whether a web search might be needed.

In [11]:
CONFIDENCE_PROMPT = ChatPromptTemplate.from_template("""
You are an evaluator. Given the following answer and context, rate:
1. Confidence score (0-10): how well the context supports the answer
2. Completeness (0-10): how fully the question was answered
3. Suggest web search: yes/no

Answer: {answer}
Context: {context}
Question: {question}

Respond in this exact format:
CONFIDENCE: <score>
COMPLETENESS: <score>
NEEDS_WEB_SEARCH: <yes/no>
REASON: <one sentence>
""")

FOLLOWUP_PROMPT = ChatPromptTemplate.from_template("""
Based on this answer, generate 3 natural follow-up questions a user might ask next:

Answer: {answer}
Original Question: {question}

Follow-up questions (one per line):
""")

def evaluated_multi_query_rag(question):
    """RAG with self-evaluation, confidence scoring, and follow-up suggestions"""
    
    # Step 1: Generate queries and retrieve docs
    query_chain = MULTI_QUERY_PROMPT | llm | StrOutputParser()
    generated_queries = query_chain.invoke({"question": question})
    queries = [q.strip() for q in generated_queries.split("\n") if q.strip()]
    
    all_docs = []
    for q in queries:
        docs = self_healing_retriever(q)
        all_docs.extend(docs[:3])
    
    # Deduplicate
    seen = set()
    unique_docs = []
    for doc in all_docs:
        if doc.page_content not in seen:
            seen.add(doc.page_content)
            unique_docs.append(doc)
    
    combined_context = safe_combine_docs(unique_docs)
    
    # Step 2: Get answer
    answer = rag_chain.invoke({
        "question": question,
        "context": combined_context
    })
    
    # Step 3: Self-evaluate confidence
    eval_chain = CONFIDENCE_PROMPT | llm | StrOutputParser()
    evaluation = eval_chain.invoke({
        "answer": answer,
        "context": combined_context[:1000],  # Truncate for eval
        "question": question
    })
    
    # Step 4: Generate follow-up questions
    followup_chain = FOLLOWUP_PROMPT | llm | StrOutputParser()
    followups = followup_chain.invoke({
        "answer": answer,
        "question": question
    })
    
    # Display results
    print(f"\n{'='*50}")
    print(f"❓ Question: {question}")
    print(f"{'='*50}")
    print(f"\n📊 Docs used: {len(unique_docs)}")
    
    print("\n---- Answer ----")
    print(answer)
    
    print("\n---- Self-Evaluation ----")
    print(evaluation)
    
    print("\n---- Suggested Follow-up Questions ----")
    print(followups)
    
    return {
        "answer": answer,
        "evaluation": evaluation,
        "followups": followups,
        "docs_used": len(unique_docs)
    }


# Test it
result = evaluated_multi_query_rag("what is the nasa sales team?")

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:langchain_classic.retrievers.multi_query:Generated queries: ['1. What details can you share about the sales department within NASA?', '2. Could you give me insights into the sales team operating within NASA?', "3. I'm interested in learning more about the sales team specifically within NASA. Can you provide relevant information?"]
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:elastic_transport.transport:POST https://8f16272b86e5488dae3f9d2a403f1070.us-central1.gcp.cloud.es.io:443/workplace-index/_search?_source_includes=metadata,text [status:200 duration:0.146s]
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:elastic_transport.transport:POST https://8f16272b86e5488dae3f9d2a403f1070.us-central1.gcp.cloud.es.io:443/work


❓ Question: what is the nasa sales team?

📊 Docs used: 3

---- Answer ----
The NASA sales team consists of two Area Vice-Presidents: Laura Martinez, who is the Area Vice-President of North America, and Gary Johnson, who is the Area Vice-President of South America.

---- Self-Evaluation ----
CONFIDENCE: 10
COMPLETENESS: 10
NEEDS_WEB_SEARCH: no
REASON: The context clearly outlines the structure of the NASA sales team and provides specific information about the Area Vice-Presidents.

---- Suggested Follow-up Questions ----
1. What are the responsibilities of the Area Vice-Presidents in the NASA sales team?
2. How many other members make up the NASA sales team besides Laura Martinez and Gary Johnson?
3. Can you provide more information on the sales strategies and goals of the NASA sales team under Laura Martinez and Gary Johnson's leadership?


## What each iteration adds

| Feature | Iteration 1 | Iteration 2 |
|---|---|---|
| Source tracking | ✅ Hit counter per source | ❌ |
| Deduplication | ✅ | ✅ |
| Multi-question loop | ✅ | ❌ |
| Confidence scoring | ❌ | ✅ |
| Follow-up suggestions | ❌ | ✅ |
| Self-evaluation | ❌ | ✅ |